# CELL-FM — NLS screening

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BoHuangLab/CELL-FM/blob/master/notebooks/nls_screening.ipynb)

**CELL-FM** generates microscopy images for unseen proteins. 
This notebook turns that into a screen for a **nuclear localization signal**.

Slide a window along a protein sequence. Hand each window's fragment to CELL-FM and ask it
to stain a cell. A fragment carrying an NLS drags the signal into the nucleus; a fragment
without one leaves it in the cytoplasm. Measuring that split, window by window, says which
stretch of the sequence carries the signal.

| Stage | What runs | Output |
| --- | --- | --- |
| **1 · Generate** | CELL-FM virtual-stains a cell for each window fragment, every one conditioned on the same anchor cell | a stack of 256×256 images per window |
| **2 · Measure** | mean generated signal inside the nucleus over mean signal in the cytoplasm | one ratio per image, one median per window |
| **3 · Screen** | those medians plotted along the sequence | the screening figure |

The nucleus and cytoplasm are read from two fixed masks drawn on the anchor cell,
so every window is measured through the same aperture and the numbers are comparable.

---

**You need a GPU runtime.** *Runtime → Change runtime type → T4 GPU*. Setup downloads about
6 GB once — 3.7 GB of CELL-FM weights and 2.3 GB for the ESM-C protein encoder.

Generation is slow: 1.66 s per image on an A40, measured, and a Colab T4 is around four
times slower again. The defaults therefore scan every eighth window rather than every one,
and the sequence cell prints a time estimate before anything runs.

| | |
| --- | --- |
| Weights | [huggingface.co/BoHuangLab/CELL-FM](https://huggingface.co/BoHuangLab/CELL-FM) |
| Code | [github.com/BoHuangLab/CELL-FM](https://github.com/BoHuangLab/CELL-FM) |


## 1 · Setup

Check the runtime, install what Colab does not ship, fetch the code, weights and anchor
cell, then build the model. Run them once, in order.


In [ ]:
import subprocess
import sys

print("python  ", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("gpu     ", gpu if gpu else
      "NONE — Runtime > Change runtime type > T4 GPU, then rerun this cell")


In [ ]:
# Colab already ships torch, numpy, pandas and matplotlib; this adds the rest of the stack.
# Two choices below look odd and are deliberate:
#
#   --no-deps on esm   its metadata requires torchtext, which has no wheel past Python
#                      3.11 and would drag torch backwards. Nothing on the ESM-C code
#                      path imports it — the second line is what the import closure
#                      actually needs, esm's other bounds included.
#   "transformers>=4.47,<4.48.2"  a range, and BOTH ends are load-bearing.
#                      4.47 is where transformers stopped assigning special tokens with
#                      setattr and started serving them through __getattr__. esm 3.2's
#                      EsmSequenceTokenizer builds cls_token, mask_token and the rest as
#                      read-only properties on top of that, so under 4.46 or older its own
#                      __init__ dies with "property 'cls_token' has no setter" — before any
#                      check in this notebook gets a chance to run.
#                      The lower bound is not redundant with the upper one. Colab
#                      preinstalls a transformers that already satisfies "<4.48.2", and pip
#                      leaves a satisfied requirement alone rather than upgrading it, so
#                      without ">=4.47" the pin silently does nothing on a fresh runtime.
#                      4.48.2 is esm's own declared bound, where its adaptation stops being
#                      tested.
#   no flash-attn      without it ESM-C falls back to a pure-torch rotary embedding,
#                      verified to give identical results, and skips a CUDA build.
#
# esm 3.1.4 used to be pinned here, and it forced a much worse install: it requires
# biotite==0.41.2, biotite 0.41 requires numpy<2, and biotite sits on the ESM-C import
# path — so the whole stack came down to NumPy 1.x. Under Colab's Python 3.13 neither
# numpy 1.26 nor biotite 0.41 has a wheel, so pip built both from source, and the numpy
# downgrade collided with every preinstalled Colab package that wants NumPy 2. esm 3.2
# moved to biotite>=1.0, which is NumPy 2 clean, and its ESM-C modules are byte-identical
# to 3.1.4's, so the checkpoint's tensor names are unchanged.
import sys

%pip install -q --no-deps --no-warn-conflicts "esm==3.2.1.post1"
%pip install -q --no-warn-conflicts "transformers>=4.47,<4.48.2" "diffusers==0.31.0" torchdiffeq loguru accelerate "biotite>=1.0" biopython msgpack-numpy cloudpathlib tenacity brotli zstd attrs einops tifffile

import importlib.metadata as md

# --no-warn-conflicts silences pip's post-install report, which here says only two things
# and both are expected:
#
#   "esm requires torchtext, which is not installed"   left unenforced on purpose by
#       --no-deps: nothing on the ESM-C code path imports it, and it has no wheel for this
#       Python. esm's transformers bound, by contrast, IS enforced above — that one is real.
#   "gradio requires huggingface-hub>=1.16"   transformers < 4.48.2 wants hub < 1.0 and
#       Colab preinstalls a gradio that wants a newer one. Nothing here imports gradio.
#
# The flag is safe here because it is not what verifies the install: the check below reads
# the installed versions, and the model cell asserts the tokenizer behaviour the pin exists
# to protect. Both are stronger than pip's declarative check, which only compares metadata.
print("\nversions")
for pkg in ("torch", "numpy", "pandas", "transformers", "diffusers", "esm", "biotite"):
    try:
        print(f"  {pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg:14s} MISSING")


def _version(pkg):
    """(major, minor, patch) for comparison; 4.48 sorts below 4.48.2, as it should."""
    return tuple(int(x) for x in md.version(pkg).split(".")[:3] if x.isdigit())


ready = (4, 47) <= _version("transformers") < (4, 48, 2)
print(f"\ntransformers {md.version('transformers')} in [4.47, 4.48.2):",
      "yes — the imports below will work" if ready else
      "NO — the next cell will fail; rerun this one")

# A pin landing on disk is not the same as this kernel using it. If an earlier attempt got
# as far as the next cell then transformers is imported, and pip downgrades it underneath a
# kernel that goes on holding the old module in memory. So compare what is imported against
# what is installed and restart if they disagree. Expected, not a crash: Colab reconnects on
# its own and you carry on from the next cell, without rerunning this one.
stale = []
for pkg in ("transformers", "tokenizers", "huggingface_hub", "biotite"):
    loaded = getattr(sys.modules.get(pkg), "__version__", None)
    try:
        installed = md.version(pkg)
    except md.PackageNotFoundError:
        continue
    if loaded is not None and loaded != installed:
        stale.append(f"{pkg} {loaded} -> {installed}")

if stale:
    import os, time
    print("\nRestarting the kernel so these take effect — expected, not a crash:")
    for line in stale:
        print("   ", line)
    print("Colab reconnects by itself; continue from the next cell.")
    time.sleep(1)
    os.kill(os.getpid(), 9)


In [ ]:
import os
import subprocess
import sys
import time

# transformers probes for a TensorFlow backend at import time and imports it if present.
# Colab ships TensorFlow, nothing here uses it, and loading it costs seconds for nothing.
# (Under the old numpy<2 pin this was a correctness fix too: TF's tflite utils import jax,
# and jax calls np.dtypes.StringDType(), a NumPy 2 API that 1.x does not have. On NumPy 2
# that no longer bites, so this is now only about import time.)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from huggingface_hub import hf_hub_download

# esm 3.2 needs transformers in [4.47, 4.48.2), and both ends bite differently. Below 4.47
# its own tokenizer cannot even be constructed — EsmSequenceTokenizer defines cls_token as
# a read-only property and the older transformers assigns special tokens with setattr, so
# building any model dies on "property 'cls_token' has no setter". At or above 4.48.2 the
# special-token plumbing moves again and mask_token comes back None, which kills generation
# later, inside esm. Check here rather than let either surface as something unrecognisable.
_tv = tuple(int(x) for x in transformers.__version__.split(".")[:3])
if not ((4, 47) <= _tv < (4, 48, 2)):
    raise RuntimeError(
        f"This kernel has transformers {transformers.__version__} loaded; esm 3.2 needs "
        ">= 4.47 and < 4.48.2. Colab preinstalls an older one and pip will not upgrade a "
        "requirement it already considers satisfied, which is why the install cell pins "
        "both ends. If it did install the right version, the kernel is still holding the "
        "old module: Runtime > Restart session, then run the cells again from the top."
    )

# The model code comes from the CELL-FM repository itself rather than from a copy: the
# notebooks and the repo cannot disagree about what the model does, and everything the repo
# has is reachable — which is how this notebook gets cell_fm.models.vit_cls, a module the
# published Space does not carry. Weights still come from the Hub.
#
# Shallow, because the history is irrelevant here and the tree packs to under a megabyte
# against the several GB of weights below. Idempotent, because Colab re-runs cells.
REPO = "https://github.com/BoHuangLab/CELL-FM.git"
CODE = os.path.join(os.getcwd(), "CELL-FM")

if not os.path.isdir(os.path.join(CODE, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True,
                   capture_output=True, text=True)
sys.path.insert(0, CODE)

# Print the commit the run actually used. The notebooks track master, so a result is only
# traceable back to code if the run says which code it was.
COMMIT = subprocess.run(["git", "-C", CODE, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

WEIGHTS_REPO = "BoHuangLab/CELL-FM"
hpa = lambda name: hf_hub_download(repo_id=WEIGHTS_REPO, filename=f"hpa/{name}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("code   ", CODE, "@", COMMIT)
print("weights", WEIGHTS_REPO, "hpa/")
print("device ", DEVICE)

# One palette for every figure below.
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
NUCLEAR, CYTO, MARK = "#2a78d6", "#eb6834", "#e53935"
plt.rcParams.update({
    "figure.dpi": 120, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
})


In [ ]:
from cell_fm.criterions.cell_fm.unidiffuser import UniDiffCriterions
from cell_fm.models.cell_fm.cell_fm_config import CELLFMConfig
from cell_fm.models.cell_fm.cell_fm_model import CELLFMModel
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer
from esm.utils import encoding

# Hyperparameters transcribed from scripts/cell_fm/evaluate_virtual_staining_hpa_dict.sh.
# infer=True is not optional: load_pretrained_weights is a no-op without it, and the model
# would run on its random initialisation rather than fail.
config = CELLFMConfig(
    img_resize=256,
    img_crop_size=1024,
    cell_image="nucl,er,mt",
    test_cell_image="nucl,er,mt",
    seq_zero_mask_ratio=0.0,
    path_type="Linear",
    prediction="velocity",
    # VAE
    num_down_blocks=3,
    latent_channels=4,
    vae_block_out_channels="128,256,512",
    # CELL-FM
    img_mask_ratio=0,
    cond_out_channels="32,64",
    sample_size=64,
    esm_embedding="esmc_600m",
    encoder_hidden_size=1152,
    max_protein_sequence_len=2048,
    encoder_num_hidden_layers=8,
    num_heads=8,
    dim_head=64,
    dropout=0,
    final_dropout=0,
    encoder_patch_size=4,
    # image generator
    img_generator_num_layers=8,
    img_generator_patch_size=2,
    attention_head_dim=64,
    num_attention_heads=18,
    # image decoder
    img_decoder_num_hidden_layers=4,
    img_decoder_hidden_size=512,
    img_decoder_num_heads=8,
    img_decoder_dim_head=64,
    # checkpoints
    vae_loadcheck_path=hpa("vae.bin"),
    loadcheck_path=hpa("cellfm_seq2img.bin"),
    infer=True,
)

# 3.7 GB of CELL-FM weights, plus 2.3 GB for the ESM-C 600M encoder the generator is built
# around — the esm package fetches that when the model is constructed, before the
# checkpoint load, even though the checkpoint carries its own copy of those tensors.
t0 = time.time()
model = CELLFMModel(config=config, loss_fn=UniDiffCriterions)
model.to(DEVICE).eval()
tokenizer = EsmSequenceTokenizer()

# The transformers pin exists to keep this true, so test it rather than trusting a version
# number. When the special-token plumbing moves under esm, mask_token comes back None and
# the failure surfaces much later, inside generation, as "replace() argument 2 must be str,
# not None". Better to hear about it here.
if not isinstance(tokenizer.mask_token, str):
    raise RuntimeError(
        f"tokenizer.mask_token is {tokenizer.mask_token!r}, not a string — this build of "
        f"transformers ({transformers.__version__}) does not serve special tokens the way "
        "esm expects, and generation would fail later. Rerun the install cell."
    )
print(f"model {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params, "
      f"ready on {DEVICE} in {time.time() - t0:.0f}s")


In [ ]:
# The anchor cell every generated image is conditioned on, and the two masks the ratio is
# read through. Both are baked assets: the anchor is HPA gene H3C13, one cell crop, run
# through the dataset's own preprocessing, and the masks are hand-drawn on that cell. Using
# one fixed cell for every window is what makes the windows comparable.
anchor = torch.from_numpy(np.load(hpa("anchor_cell.npy"))).unsqueeze(0).to(DEVICE)
_masks = np.load(hpa("anchor_masks.npz"))
NUCLEUS, CELL_BODY = _masks["nucleus"], _masks["cell"]
CYTOPLASM = CELL_BODY & ~NUCLEUS

print(f"anchor {tuple(anchor.shape)} in [{anchor.min():.1f}, {anchor.max():.1f}] "
      "— channels nucleus, ER, microtubules")
print(f"masks  nucleus {NUCLEUS.sum():,} px, cytoplasm {CYTOPLASM.sum():,} px")


def nuclear_ratio(images01):
    """Mean signal in the nucleus over mean signal in the cytoplasm, one value per image."""
    inside = images01[:, NUCLEUS].mean(axis=1)
    outside = images01[:, CYTOPLASM].mean(axis=1)
    return inside / (outside + 1e-8)

titles = ["nucleus", "ER", "microtubules"]
fig, axes = plt.subplots(1, 5, figsize=(11.5, 2.4))
for ax, channel, title in zip(axes, anchor[0].cpu().numpy(), titles):
    ax.imshow((channel + 1) / 2, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=8)
axes[3].imshow(NUCLEUS, cmap="gray")
axes[3].set_title("nucleus mask", fontsize=8, color=NUCLEAR)
axes[4].imshow(CYTOPLASM, cmap="gray")
axes[4].set_title("cytoplasm mask", fontsize=8, color=CYTO)
for ax in axes:
    ax.set_axis_off()
fig.suptitle("the anchor cell, and the aperture every window is measured through",
             fontsize=10, y=1.04)
plt.show()


## 2 · Choose a sequence and a window

The sliding window is the screen. Each window contributes one point to the final figure,
and each point costs `IMAGES_PER_WINDOW` generated images, so the cost is
`windows x images`. `STRIDE` is how the run is kept to a few minutes: at 1 every window is
scanned, at 4 every fourth.

Cut with `STRIDE` before cutting `IMAGES_PER_WINDOW`. The per-window number is a median
over the images, and a median over 8 samples is a much shakier quantity than a median over
32 — better a sparser curve made of solid points than a dense one made of noise.

**This is an expensive model.** 

The figure marks PRRSV's two known NLS motifs in red under the axis, so the default run
shows straight away whether the peaks land on them. They are dropped as soon as you paste a
different sequence; to mark motifs on your own protein, set `nls_ranges` in the cell below
to ranges like `[(10, 13), (41, 47)]`.

In [ ]:
# @title Sequence and window { display-mode: "form" }
# The form is the whole interface, and it asks only for what changes a run. The default is
# the PRRSV nucleocapsid protein; its two known NLS motifs are marked on the figure so the
# first run has something to check against.
SEQUENCE = "MPNNNGKQQKRKKGDGQPVNQLCQMLGKIIAQQNQSRGKGPGKKNKKKNPEKPHFPLATEDDVRHHFTPSERQLCLSSIQTAFNQGAGTCTLSDSGRISYTVEFSLPTHHTVRLIRVTASPSA"  # @param {type:"string"}
WINDOW = 25  # @param {type:"integer"}
STRIDE = 2  # @param {type:"integer"}
IMAGES_PER_WINDOW = 32  # @param {type:"integer"}
SEED = 6  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}

# Not in the form. The ODE step count is the sampler setting the timings below assume, and
# lowering it trades image quality for speed close to linearly.
NUM_STEPS = 100

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")


def clean_sequence(text):
    """Strip FASTA headers, whitespace and case; refuse anything but standard residues."""
    seq = "".join(l for l in str(text).splitlines() if not l.startswith(">"))
    seq = "".join(seq.split()).upper()
    if not seq:
        raise ValueError("Enter a protein sequence.")
    bad = sorted(set(seq) - VALID_AA)
    if bad:
        raise ValueError(f"Sequence contains non-standard residues: {', '.join(bad)}.")
    return seq


def fasta_name(text):
    """The name from a FASTA header, when the sequence is pasted with one."""
    for line in str(text).splitlines():
        if line.startswith(">"):
            return line[1:].strip().split("|")[-1].strip()[:60] or None
    return None


def parse_ranges(text):
    """'10-13, 41-47' -> [(10, 13), (41, 47)]; blank is fine and means no annotation."""
    out = []
    for part in str(text).replace(",", " ").split():
        lo, _, hi = part.partition("-")
        try:
            out.append((int(lo), int(hi or lo)))
        except ValueError:
            raise ValueError(f"nls_ranges: cannot read {part!r}. "
                             "Use residue numbers, like '10-13, 41-47'.") from None
    return out


sequence = clean_sequence(SEQUENCE)

# Both of these used to be form fields, and both describe a particular protein rather than
# a run. Left as free text they outlive the sequence they belong to: paste something else
# and the figure still says PRRSV and still marks PRRSV's motifs in red, on residues that
# have nothing to do with them. So the name comes from a FASTA header when you paste one,
# and the NLS marks apply only to the sequence that ships with the notebook.
is_prrsv = sequence.startswith("MPNNNGKQQKRKK") and len(sequence) == 123
PROTEIN_NAME = fasta_name(SEQUENCE) or ("PRRSV" if is_prrsv else "Protein")
nls_ranges = parse_ranges("10-13, 41-47") if is_prrsv else []

if WINDOW >= len(sequence):
    raise ValueError(f"WINDOW ({WINDOW}) must be shorter than the sequence ({len(sequence)} aa).")

# Windows match the offline task: fragment = "M" + sequence[i:i+WINDOW], named by the
# 1-indexed residue range it covers. STRIDE just thins that same list.
windows = [
    {"name": f"{i + 1}-{i + WINDOW}", "start": i + 1, "end": i + WINDOW,
     "fragment": "M" + sequence[i:i + WINDOW]}
    for i in range(1, len(sequence) - WINDOW + 1, STRIDE)
]

print(f"{PROTEIN_NAME}: {len(sequence)} aa")
print(f"window {WINDOW}, stride {STRIDE} -> {len(windows)} windows "
      f"({len(range(1, len(sequence) - WINDOW + 1))} at stride 1)")
# Measured on an A40, batch 16, 100 ODE steps. A T4 is roughly four times slower; a
# smaller NUM_STEPS scales this close to linearly.
SECONDS_PER_IMAGE = 1.66
n_images = len(windows) * IMAGES_PER_WINDOW
print(f"{len(windows)} x {IMAGES_PER_WINDOW} = {n_images} images to generate")
print(f"first {windows[0]['name']}  last {windows[-1]['name']}")
print(f"\nestimate: {n_images * SECONDS_PER_IMAGE / 60:.0f} min on an A40, "
      f"~{n_images * SECONDS_PER_IMAGE * 4 / 60:.0f} min on a T4. "
      "Raise STRIDE to cut it.")
# Two residues can fall outside the screen and both are easy to miss. Window 1 starts at
# residue 2, because the fragment prepends its own M and starting at 1 would double the
# native one. And at a wide STRIDE the last window can stop short of the C-terminus.
covered = set()
for w in windows:
    covered |= set(range(w["start"], w["end"] + 1))
unscanned = sorted(set(range(1, len(sequence) + 1)) - covered)
if unscanned:
    runs, lo = [], unscanned[0]
    for a, b in zip(unscanned, unscanned[1:] + [None]):
        if b != a + 1:
            runs.append(str(lo) if lo == a else f"{lo}-{a}")
            lo = b
    print(f"not scanned: residue {', '.join(runs)} "
          f"({len(unscanned)} of {len(sequence)}) — lower STRIDE to close the gap")

if nls_ranges:
    print("annotating: " + ", ".join(f"{lo}-{hi} ({sequence[lo - 1:hi]})" for lo, hi in nls_ranges))


## 3 · Stage 1 — generate a cell per window

Every fragment is stained against the same anchor cell, and sampling is seeded per window,
so two runs of the same window give the same images and two different windows differ only
by their fragment.

In [ ]:
import random

from tqdm.auto import tqdm


def seed_everything(seed):
    """Matches cell_fm/pipeline/accelerator/trainer.py, which the offline task calls."""
    torch.cuda.manual_seed_all(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


@torch.no_grad()
def generate(fragment, n_images, seed=SEED):
    """Virtual-stain n_images cells for one fragment, returned as (n, 256, 256) in [0, 1]."""
    tokens = encoding.tokenize_sequence(fragment, tokenizer, True).unsqueeze(0).to(DEVICE)
    seed_everything(seed)

    out = []
    for start in range(0, n_images, BATCH_SIZE):
        b = min(BATCH_SIZE, n_images - start)
        sample = model.sequence_to_image(
            tokens.repeat(b, 1), anchor.repeat(b, 1, 1, 1), num_steps=NUM_STEPS)
        out.append(((sample[:, 0] + 1) / 2).clamp(0, 1).cpu().numpy())
    return np.concatenate(out)


if str(DEVICE) == "cpu":
    print("No GPU — this will take many hours. Runtime > Change runtime type > T4 GPU.\n")

# Each window is measured as soon as it is generated and then all but a few of its images
# are dropped. Holding every image would cost len(windows) x IMAGES_PER_WINDOW x 256 x 256
# floats -- 1.6 GB at stride 1 with 64 images, enough to end a Colab session -- and nothing
# downstream needs more than the ratios and a few pictures.
SAMPLES_KEPT = 6

t0 = time.time()
for w in tqdm(windows, desc="windows", unit="win"):
    images = generate(w["fragment"], IMAGES_PER_WINDOW)
    w["ratios"] = nuclear_ratio(images)
    w["median_ratio"] = float(np.median(w["ratios"]))
    # the images closest to this window's median: the ones that represent it honestly
    keep = np.argsort(np.abs(w["ratios"] - w["median_ratio"]))[:SAMPLES_KEPT]
    w["samples"] = images[keep]
gen_seconds = time.time() - t0

total = len(windows) * IMAGES_PER_WINDOW
print(f"{total} images in {gen_seconds / 60:.1f} min — {gen_seconds / total:.1f}s per image")
print(f"kept {SAMPLES_KEPT} images per window, "
      f"{sum(w['samples'].nbytes for w in windows) / 1e6:.0f} MB")


## 4 · Stage 2 — read the measurements

Each window was scored as it was generated: for every image, the mean signal inside the
nucleus mask over the mean signal in the cytoplasm mask, and for the window, the **median**
of those.

In [ ]:
ranked = sorted(windows, key=lambda w: w["median_ratio"])
most_cyto, most_nuclear = ranked[0], ranked[-1]
print(f"most nuclear    {most_nuclear['name']}  ratio {most_nuclear['median_ratio']:.2f}"
      f"  {most_nuclear['fragment'][1:]}")
print(f"most cytoplasmic {most_cyto['name']}  ratio {most_cyto['median_ratio']:.2f}"
      f"  {most_cyto['fragment'][1:]}")

# the two extremes side by side, same greyscale, so the ratio is visible rather than asserted
fig, axes = plt.subplots(2, 6, figsize=(11.5, 4.2))
for row, (w, colour) in enumerate(((most_nuclear, NUCLEAR), (most_cyto, CYTO))):
    for ax in axes[row]:
        ax.set_axis_off()
    for ax, img in zip(axes[row], w["samples"]):
        ax.imshow(img, cmap="magma", vmin=0, vmax=1)
    axes[row][0].text(-0.12, 0.5, f"{w['name']}\nratio {w['median_ratio']:.2f}",
                      transform=axes[row][0].transAxes, ha="right", va="center",
                      fontsize=9, color=colour, fontweight="bold")
fig.suptitle("most nuclear window (top) against most cytoplasmic (bottom), same scale",
             fontsize=10, y=1.02)
plt.show()


## 5 · Stage 3 — the screen

One point per window, in sequence order. Peaks are windows whose fragment carried the
protein into the nucleus.

In [ ]:
def plot_screen(windows, sequence, protein_name, nls_ranges=(), ax=None):
    """The screening figure: median nuclear/cytoplasmic ratio along the sequence."""
    names = [w["name"] for w in windows]
    values = [w["median_ratio"] for w in windows]
    gap = 10
    xs = [i * gap for i in range(len(windows))]

    if ax is None:
        width = max(7.0, min(26.0, 0.24 * len(windows) + 3.0))
        _, ax = plt.subplots(figsize=(width, 6))

    # The band is the spread across the images within each window — the same quartiles
    # the CSV carries. Medians alone look far more settled than the images actually are.
    spread = all("ratios" in w for w in windows)
    if spread:
        q1 = [float(np.percentile(w["ratios"], 25)) for w in windows]
        q3 = [float(np.percentile(w["ratios"], 75)) for w in windows]
        ax.fill_between(xs, q1, q3, color=INK, alpha=0.10, linewidth=0, zorder=1)

    ax.plot(xs, values, color=INK, linewidth=1.2, zorder=3)
    ax.scatter(xs, values, s=50, color=INK, zorder=4)
    ax.axhline(1.0, color=MUTED, linewidth=1.0, linestyle=":", zorder=2)

    ax.set_xticks(xs)
    ax.set_xticklabels(names, rotation=90,
                       fontsize=max(5, min(15, 900 / max(1, len(windows)))))
    ax.set_xlim(xs[0] - gap, xs[-1] + gap)
    # Scale to the data while keeping the ratio = 1 reference inside the axis. A fixed
    # ceiling flattens any protein whose ratios sit near 1 — on a PRRSV-scale screen it
    # spent under a third of the axis on the data.
    ax.set_ylim(0, max(1.4, (max(q3) if spread else max(values)) * 1.15))
    ax.set_ylabel("Median intensity ratio\n(nucleus / cytoplasm)", fontsize=13)
    # The old title spelled out "nuclear vs. cytoplasmic intensity", which the y-label
    # already says, and at that length it overran the figure -- a left-aligned title is
    # not pulled back by tight_layout. Shorter, and it leaves room for a long name.
    title = ax.set_title(f"{protein_name}: NLS screen across sequence windows",
                         fontsize=14, loc="left")
    ax.grid(axis="y", linestyle="--", linewidth=0.5, color="#eeeeee", zorder=0)
    ax.set_axisbelow(True)

    # PROTEIN_NAME is free text and a left-aligned title that overruns the figure is not
    # pulled back by tight_layout, so measure it and shrink until it fits. The margin
    # leaves room for the shift tight_layout applies afterwards for the y-label.
    fig = ax.figure
    fig.canvas.draw()
    budget = fig.get_size_inches()[0] * fig.dpi - 90
    for _ in range(10):
        if title.get_window_extent(fig.canvas.get_renderer()).x1 <= budget:
            break
        title.set_fontsize(title.get_fontsize() * 0.9)
        fig.canvas.draw()

    # The residue row. Window i is centred on residue mid_i, and consecutive windows step by
    # STRIDE residues, so residue p belongs at x = gap * (p - mid_0) / STRIDE. At stride 1
    # that places every residue, including the ones before the first window's midpoint and
    # after the last; at wider strides only the midpoints get a letter.
    stride = (windows[1]["start"] - windows[0]["start"]) if len(windows) > 1 else 1
    mid = lambda w: (w["start"] + w["end"]) // 2
    if stride == 1:
        positions = range(1, len(sequence) + 1)
    else:
        positions = [mid(w) for w in windows]

    trans = ax.get_xaxis_transform()
    slot = (ax.figure.get_size_inches()[0] * 72) / max(1, len(list(positions)))
    size = max(4.0, min(18.0, slot * 0.95))
    for p in positions:
        x = gap * (p - mid(windows[0])) / stride
        red = any(lo <= p <= hi for lo, hi in nls_ranges)
        ax.text(x, -0.28, sequence[p - 1], transform=trans, ha="center", va="top",
                fontsize=size, color=MARK if red else INK, clip_on=False)
    return ax


plot_screen(windows, sequence, PROTEIN_NAME, nls_ranges)
plt.tight_layout()
plt.show()

by_ratio = sorted(windows, key=lambda w: w["median_ratio"])
peak, runner_up = by_ratio[-1], by_ratio[-2]
q1, q3 = np.percentile(peak["ratios"], [25, 75])
print(f"peak at {peak['name']}, ratio {peak['median_ratio']:.2f} (IQR {q1:.2f}-{q3:.2f}) "
      f"— residues {sequence[peak['start'] - 1:peak['end']]}")
# The peak is the argmax of a few noisy medians, so it is the most optimistic window
# rather than a measured winner. If the runner-up sits inside the peak's IQR, they are
# not separated by this run.
print(f"next highest {runner_up['name']} at {runner_up['median_ratio']:.2f}"
      + ("  — within the peak's IQR, so this run does not separate them"
         if runner_up["median_ratio"] >= q1 else ""))
print(f"{sum(w['median_ratio'] > 1 for w in windows)} of {len(windows)} windows are nuclear "
      "(ratio > 1)")


## 6 · Export

Two files, each with a button to download it.

| File | What it holds |
| --- | --- |
| `nls_screen.tif` | one representative image per window as `uint16`, the one whose ratio is closest to that window's median, tagged so ImageJ opens it as a stack in sequence order |
| `nls_screen.csv` | one row per window: the residue range, the fragment, the median ratio and the spread across images |


In [ ]:
import tifffile


def export(windows, sequence, protein_name, stem="nls_screen"):
    """Write the per-window stack and table, then offer both for download."""
    tif_path, csv_path = f"{stem}.tif", f"{stem}.csv"

    stack = np.stack([w["samples"][0] for w in windows])
    tifffile.imwrite(tif_path, (stack * 65535).astype(np.uint16), imagej=True,
                     metadata={"axes": "ZYX"})

    pd.DataFrame([{
        "window": w["name"],
        "start": w["start"],
        "end": w["end"],
        "midpoint_residue": (w["start"] + w["end"]) // 2,
        "fragment": w["fragment"],
        "median_ratio": w["median_ratio"],
        "ratio_q1": float(np.percentile(w["ratios"], 25)),
        "ratio_q3": float(np.percentile(w["ratios"], 75)),
        "n_images": len(w["ratios"]),
        "protein": protein_name,
    } for w in windows]).to_csv(csv_path, index=False)

    download_buttons(tif_path, csv_path)
    return tif_path, csv_path


def download_buttons(*paths):
    """One button per file, rather than firing the downloads the moment the cell runs.

    The Output widget is load-bearing: files.download runs JavaScript in a live output
    context, and a click handler has none of its own, so calling it straight from on_click
    silently does nothing.
    """
    for path in paths:
        print(f"{path}  {os.path.getsize(path) / 1e6:.1f} MB")

    try:
        import ipywidgets as widgets
        from google.colab import files
        from IPython.display import display
    except ImportError:
        print("\nno download button outside Colab — both files are in the working directory")
        return

    sink = widgets.Output()

    def fetch(path):
        with sink:
            files.download(path)

    buttons = [
        widgets.Button(description=f"Download {os.path.basename(path)}",
                       icon="download", layout=widgets.Layout(width="auto"))
        for path in paths
    ]
    for button, path in zip(buttons, paths):
        button.on_click(lambda _, p=path: fetch(p))

    display(widgets.HBox(buttons), sink)


_ = export(windows, sequence, PROTEIN_NAME)
